# Solutions generation

Generating solutions of ill-defined problems using different LLM-based baselines.

In [20]:
PROBLEMS_FILE = './data/problems.json'
SOLUTIONS_FILE = './data/solutions.json'
MAX_WORKERS = 5
TEMPERATURE = 0.5
MODELS = [
    "deepseek/deepseek-v4-pro", # strong frontier
    "openai/gpt-4o-mini", # mid strong / efficient
    "microsoft/phi-4-mini-instruct", # lower-mid anchor
]

In [21]:
from src import read_json, write_json, generate_id

problems = read_json(PROBLEMS_FILE)
solutions = read_json(SOLUTIONS_FILE) or {}

In [22]:
from langchain_openai import ChatOpenAI
from fp2mp_eval._config import config

def init_llm(model : str):
    return ChatOpenAI(
        model=model,
        base_url=config.base_url,
        api_key=config.api_key,
        temperature=TEMPERATURE,
    )

Creating tasks for problems-baselines pairs not presented in `solutions.json`

In [23]:
tasks = []

for problem_id in problems.keys():
    for model in MODELS:
        solution_id = generate_id(problem_id, model)
        if solution_id not in solutions:
            tasks.append({
                'solution_id': solution_id,
                'problem_id': problem_id,
                'model': model
            })

len(tasks)

84

In [24]:
from tqdm.contrib.concurrent import thread_map

def solve_task(task : dict[str,str]):
    solution_id = task['solution_id']
    problem_id = task['problem_id']
    model = task['model']

    problem_content = problems[problem_id]['content']
    llm = init_llm(model)

    response = llm.invoke(problem_content)
    return {
        'solution_id': solution_id,
        'solution_data': {
            'problem_id': problem_id,
            'model': model,
            'content': response.content,
        }
    }    

results = thread_map(solve_task, tasks, max_workers=MAX_WORKERS)

  0%|          | 0/84 [00:00<?, ?it/s]

In [25]:
for result in results:
    solution_id = result['solution_id']
    solution_data = result['solution_data']
    solutions[solution_id] = solution_data

write_json(solutions, SOLUTIONS_FILE)